# Role 2 — BIMBA Baseline: Video-Language Model Inference & Evaluation

**Project:** ChronoMind — ReVA Drone Video Understanding Challenge (WACV 2027)  
**Branch:** `rutvij`  
**Target Environment:** Google Colab (Tesla T4 GPU, ~15 GB VRAM)  
**Model Architecture:** BIMBA (Selective-Scan Token Compression with Mamba on LLaVA-Video-7B-Qwen2)  
- Base Model: `lmms-lab/LLaVA-Video-7B-Qwen2` (rev: `013210b3aff822f1558b166d39c1046dd109520f`)
- Adapter: `mmiemon/BIMBA-LLaVA-Qwen2-7B` (rev: `f09f08dcd7401410531d2bb85f5f8c7208821981`)
- Challenge: 4-way Multiple Choice Drone Video Question Answering

## 1. Environment Verification & GPU Preflight Check
Verify that an NVIDIA GPU (Tesla T4, 15 GB VRAM) is allocated, check system paths, and verify working directory.

In [ ]:
import os
import sys
from pathlib import Path

# Resolve workspace directory
WORKSPACE_DIR = Path('/content/chronomind')
if not WORKSPACE_DIR.is_dir():
    if Path('/content/ChronoMind').is_dir():
        WORKSPACE_DIR = Path('/content/ChronoMind')
    else:
        WORKSPACE_DIR = Path.cwd()

sys.path.insert(0, str(WORKSPACE_DIR))
sys.path.insert(0, str(WORKSPACE_DIR / 'Evaluation_Framework' / 'src'))
print(f"Workspace root: {WORKSPACE_DIR}")

# Check CUDA device
try:
    import torch
    print("CUDA Available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        dev_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"Active GPU: {dev_name} ({vram:.2f} GB VRAM)")
    else:
        print("WARNING: CUDA is not available. Connect to a GPU runtime (T4).")
except ImportError:
    print("PyTorch not yet installed.")

## 2. Pinned Environment Installation
Install the verified dependency stack:
- PyTorch 2.1.2 + cu121
- Transformers 4.40.0.dev0
- bitsandbytes 0.43.3, Accelerate 0.29.3, PEFT 0.10.0
- decord 0.6.0, av 17.1.0
- causal-conv1d 1.4.0 & mamba-ssm 2.2.2 (Mamba CUDA kernels)
- pydantic>=2.0, pandas, pyyaml, matplotlib (Universal Evaluation Framework)

In [ ]:
# Execute in Colab
!pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.40.0.dev0 accelerate==0.29.3 bitsandbytes==0.43.3 peft==0.10.0 decord==0.6.0 av==17.1.0
!pip install -q "pydantic>=2.0" pandas pyyaml matplotlib jsonschema

# Install Mamba CUDA extensions without build isolation
!pip install -q causal-conv1d==1.4.0 --no-build-isolation
!pip install -q mamba-ssm==2.2.2 --no-build-isolation

## 3. Clone Official BIMBA Repository & Verify CUDA Extensions
Clone `md-mohaiminul/BIMBA` to import custom model architecture modules.

In [ ]:
import subprocess

BIMBA_REPO_DIR = Path("/content/BIMBA")
if not BIMBA_REPO_DIR.is_dir():
    print("Cloning official BIMBA repository...")
    subprocess.run(["git", "clone", "https://github.com/md-mohaiminul/BIMBA.git", str(BIMBA_REPO_DIR)], check=True)
else:
    print(f"BIMBA repo found at {BIMBA_REPO_DIR}")

sys.path.insert(0, str(BIMBA_REPO_DIR))

# Verify CUDA build
import causal_conv1d_cuda
import selective_scan_cuda
print("Mamba CUDA extensions loaded successfully.")

## 4. Model Loading with 4-Bit NF4 Quantization & Dtype Diagnosis
Load `lmms-lab/LLaVA-Video-7B-Qwen2` with 4-bit NF4 quantization for the 15 GB Tesla T4.
Inspect the first Transformer attention layer (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `lora_A`, `lora_B`), diagnose the float32/float16 divergence, and apply the targeted precision alignment fix.

In [ ]:
from baseline.bimba_inference import (
    load_bimba_for_inference,
    diagnose_model_dtypes,
    apply_precision_alignment,
    BASE_MODEL,
    ADAPTER_MODEL,
)

print(f"Loading Base: {BASE_MODEL}")
print(f"Loading Adapter: {ADAPTER_MODEL}")

tokenizer, model, image_processor, context_len = load_bimba_for_inference(
    repo_path=BIMBA_REPO_DIR,
    base_model=BASE_MODEL,
    adapter_model=ADAPTER_MODEL,
    load_in_4bit=True,
    device="cuda",
    apply_dtype_fix=True,
)

# Run post-load diagnostic
diag = diagnose_model_dtypes(model)
print("\n--- Parameter Dtype Summary ---")
print("Overall dtypes:", diag["overall_dtype_summary"])
print("Dtype mismatch detected:", diag["dtype_mismatch_detected"])
if diag["divergent_modules"]:
    print("Divergent modules:", diag["divergent_modules"])
else:
    print("All attention projection and LoRA layers are aligned to float16.")

## 5. Download Sample Benchmark Video (`DJI_0157_d4_01.mp4`)
Download the exact sample video from the ReVA benchmark on Hugging Face using the manifest utility.

In [ ]:
SAMPLE_VIDEO_REL = "Hawk_UAV/NJ/DJI_0157_d4_01.mp4"

# Download via dataset utility directly into workspace
!python scripts/download_dataset.py --filename "{SAMPLE_VIDEO_REL}" --local-dir .

# Resolve downloaded file
LOCAL_VIDEO_PATH = Path(SAMPLE_VIDEO_REL)
if not LOCAL_VIDEO_PATH.is_file():
    import glob
    matches = glob.glob(f"**/{Path(SAMPLE_VIDEO_REL).name}", recursive=True)
    if matches:
        LOCAL_VIDEO_PATH = Path(matches[0])

print(f"Sample Video: {LOCAL_VIDEO_PATH} (Exists: {LOCAL_VIDEO_PATH.is_file()})")


## 6. One-Video Inference Smoke Test (`val_000001`)
Run inference on sample `val_000001`:
- **Question:** "At what point in the video does the road with moving vehicles first appear?"
- **Choices:** A. 5.0s, B. 4.2s, C. 2.5s, D. 3.8s
- **Ground Truth:** B

Verify that generation succeeds without the Qwen2 attention dtype crash, and measure latency and peak GPU memory.

In [ ]:
import torch
from baseline.bimba_inference import predict_single_sample, _save_canonical_json
from reva_eval.evaluator import evaluate

torch.cuda.reset_peak_memory_stats()

sample_q = "At what point in the video does the road with moving vehicles first appear?"
sample_opts = {
    "A": "At 5.0 seconds",
    "B": "At 4.2 seconds",
    "C": "At 2.5 seconds",
    "D": "At 3.8 seconds",
}

print("Executing single-sample inference...")
res = predict_single_sample(
    model=model,
    tokenizer=tokenizer,
    image_processor=image_processor,
    video_path=LOCAL_VIDEO_PATH,
    question=sample_q,
    options=sample_opts,
    sample_id="val_000001",
    num_frames=16,
)

peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

print("\n=======================================================")
print("            SMOKE TEST INFERENCE RESULT                ")
print("=======================================================")
print(f"Sample ID:             {res['sample_id']}")
print(f"Predicted Choice:      {res['prediction']}")
print(f"Raw Generated Text:    {repr(res['raw_text'])}")
print(f"Normalized Scores:     {res['scores']}")
print(f"Inference Latency:     {res['inference_time_seconds']:.2f} s")
print(f"Peak GPU Memory:       {peak_vram:.2f} GB")
print(f"Expected Ground Truth: B")
if res['prediction'] == 'B':
    print("STATUS: VERIFIED CORRECT PREDICTION!")
else:
    print(f"STATUS: Inference completed (Prediction: {res['prediction']})")
print("=======================================================")

# Export single prediction to canonical JSON format for reva_eval
SMOKE_OUT_JSON = WORKSPACE_DIR / "reports" / "predictions_smoke.json"
_save_canonical_json(
    records=[{
        "sample_id": res['sample_id'],
        "prediction": res['prediction'],
        "scores": res['scores'],
        "inference_time_seconds": res['inference_time_seconds'],
    }],
    output_path=SMOKE_OUT_JSON,
    model_name="BIMBA-LLaVA-Qwen2-7B",
    model_version="CVPR2025",
    run_id="bimba_smoke_test",
)
print(f"\nSaved canonical smoke prediction JSON to: {SMOKE_OUT_JSON}")

# Run single-sample evaluation through reva_eval
smoke_eval = evaluate(
    ground_truth=WORKSPACE_DIR / "Evaluation_Framework" / "data" / "annotations" / "val.json",
    predictions=SMOKE_OUT_JSON,
    output_dir=WORKSPACE_DIR / "reports" / "smoke_evaluation",
    save_plots=False,
    overwrite=True,
)
print(f"Evaluation report generated at: {WORKSPACE_DIR / 'reports' / 'smoke_evaluation'}")


## 7. Resumable Validation Split Inference
Run validation inference across the canonical validation set (`Evaluation_Framework/data/annotations/val.json`).
Exports canonical predictions matching `Evaluation_Framework/schemas/predictions.schema.json`.

In [ ]:
from baseline.bimba_inference import run_batch_inference

VAL_ANNOTATION = WORKSPACE_DIR / "Evaluation_Framework" / "data" / "annotations" / "val.json"
OUTPUT_PREDICTIONS = WORKSPACE_DIR / "reports" / "predictions_bimba_val.json"
VIDEO_DIR = Path("/content/reva_video_cache")

# Set subset size (e.g. 20 for initial benchmark, or None for full 2,000 samples)
SUBSET = 20

run_batch_inference(
    annotation_path=VAL_ANNOTATION,
    video_dir=VIDEO_DIR,
    output_path=OUTPUT_PREDICTIONS,
    repo_path=BIMBA_REPO_DIR,
    subset=SUBSET,
    num_frames=16,
    resume=True,
)

## 8. Evaluation via Universal Evaluation Framework (`reva_eval`)
Score the generated predictions against canonical ground truth.

In [ ]:
from reva_eval.evaluator import evaluate

EVAL_OUT_DIR = WORKSPACE_DIR / "reports" / "bimba_val_evaluation"

print("Running ReVA evaluation...")
report = evaluate(
    ground_truth=VAL_ANNOTATION,
    predictions=OUTPUT_PREDICTIONS,
    output_dir=EVAL_OUT_DIR,
    save_plots=True,
    overwrite=True,
)

print("\n=======================================================")
print("            BIMBA BASELINE EVALUATION REPORT           ")
print("=======================================================")
print(f"Model:                    {report.model.name}")
print(f"Status:                   {report.status}")
print(f"Evaluated Samples:        {report.metrics.provided_predictions}")
print(f"Valid Coverage:           {report.metrics.valid_prediction_coverage:.2%}")
if report.ground_truth_labeled and report.metrics.accuracy is not None:
    print(f"Overall Accuracy:         {report.metrics.accuracy:.2%} ({report.metrics.correct}/{report.metrics.total_samples})")
print(f"Mean Latency:             {report.runtime.mean_seconds:.2f} s")
print("=======================================================")